In [1]:
import numpy as np
from IPython.display import Video, display, clear_output
import ipywidgets as widgets
from manim import *
import tempfile
import os

# 设置 Manim 渲染质量（预览用低质量可加速）
config.quality = "low_quality"   # 480p 15fps
config.media_dir = tempfile.mkdtemp()   # 输出到临时目录
config.disable_caching = True          # 禁用缓存，保证每次重新渲染
# video_path = config.media_dir + "/videos/480p15/MatrixRotation.mp4"

In [2]:
class MatrixRotation(Scene):
    """根据传入角度渲染矩阵旋转动画"""
    def __init__(self, angle_deg=45, **kwargs):
        super().__init__(**kwargs)
        self.angle_deg = angle_deg

    def construct(self):
        angle = self.angle_deg * DEGREES
        rot_matrix = np.array([
            [np.cos(angle), -np.sin(angle)],
            [np.sin(angle),  np.cos(angle)]
        ])

        # 原始向量
        original_vec = np.array([2, 0, 0])   # 在 xy 平面
        arrow_orig = Arrow(start=ORIGIN, end=original_vec, color=BLUE, buff=0)
        label_orig = MathTex(r"[1,0]").next_to(arrow_orig.get_end(), UR)

        # 旋转后的向量
        rotated_vec = np.array([* (rot_matrix @ original_vec[:2]), 0])
        arrow_rot = Arrow(start=ORIGIN, end=rotated_vec, color=RED, buff=0)
        label_rot = MathTex(r"R_\theta [1,0]").next_to(arrow_rot.get_end(), UR)

        # 旋转矩阵文本
        matrix_tex = MathTex(
            r"R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}"
        ).to_corner(UL)
        angle_tex = MathTex(r"\theta = {:.1f}^\circ".format(self.angle_deg)).next_to(matrix_tex, DOWN)

        # 动画
        self.add(matrix_tex, angle_tex)
        self.play(GrowArrow(arrow_orig), Write(label_orig))
        self.wait(0.3)
        self.play(
            Rotate(arrow_orig, angle=angle, about_point=ORIGIN),
            Transform(label_orig, label_rot),
            run_time=2
        )
        self.wait(0.5)

In [3]:
def render_and_show(angle):
    """根据角度渲染动画，并在输出区域显示视频"""
    # 清除之前的输出
    clear_output(wait=True)
    
    # 使用自定义场景渲染
    scene = MatrixRotation(angle_deg=angle)
    scene.render()
    
    # 获取生成的视频文件（Manim 默认用场景类名保存）
    # 注意文件名中包含场景类名和配置质量
    video_path = config.media_dir + "/videos/480p15/MatrixRotation.mp4"
    
    if os.path.exists(video_path):
        display(Video(video_path, embed=True, width=600))
    else:
        print(f"视频未找到: {video_path}")

# 创建滑块（0° ~ 360°，默认45°）
angle_slider = widgets.FloatSlider(
    value=45,
    min=0,
    max=360,
    step=1,
    description='旋转角度',
    continuous_update=False  # 只在松开滑块时触发，避免频繁渲染
)

# 交互输出
out = widgets.interactive_output(render_and_show, {'angle': angle_slider})
display(angle_slider, out)

FloatSlider(value=45.0, continuous_update=False, description='旋转角度', max=360.0, step=1.0)

Output()

In [5]:
## 向量旋转、矩阵旋转、坐标系旋转与坐标系变换

# 下面我们从四个角度理解"旋转"这一核心概念。

### 1. 向量旋转 (Vector Rotation)

向量旋转是**主动**旋转：向量本身绕原点旋转 $\theta$ 角度，坐标系不动。

旋转公式：
$$v' = R_\theta \cdot v$$

其中旋转矩阵：
$$R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

下面展示多个不同方向的向量同时旋转同一角度：

In [6]:
class VectorRotationScene(Scene):
    """向量旋转：多个向量同时旋转同一角度"""
    def __init__(self, angle_deg=45, **kwargs):
        super().__init__(**kwargs)
        self.angle_deg = angle_deg

    def construct(self):
        angle = self.angle_deg * DEGREES
        rad = np.radians(self.angle_deg)
        rot_matrix = np.array([
            [np.cos(rad), -np.sin(rad)],
            [np.sin(rad),  np.cos(rad)]
        ])

        # 原始向量列表 (x, y, 颜色, 标签)
        vectors = [
            (np.array([2, 0]), BLUE, r"\mathbf{v}_1"),
            (np.array([0, 1.5]), GREEN, r"\mathbf{v}_2"),
            (np.array([1.5, 1]), YELLOW, r"\mathbf{v}_3"),
            (np.array([-1.5, 1.2]), PURPLE, r"\mathbf{v}_4"),
        ]

        # 坐标轴
        axes = Axes(
            x_range=[-3, 3, 1], y_range=[-3, 3, 1],
            x_length=6, y_length=6,
            axis_config={"include_tip": True, "color": WHITE}
        )
        self.add(axes)

        # 公式
        formula = MathTex(
            r"\mathbf{v}' = R_\theta \mathbf{v}, \quad "
            r"R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ "
            r"\sin\theta & \cos\theta \end{bmatrix}"
        ).to_corner(UL).scale(0.7)
        angle_label = MathTex(
            r"\theta = {:.1f}^\circ".format(self.angle_deg)
        ).next_to(formula, DOWN, aligned_edge=LEFT)
        self.add(formula, angle_label)

        # 画原始向量
        arrows_orig = []
        labels_orig = []
        for vec, color, label_tex in vectors:
            end = axes.c2p(vec[0], vec[1])
            arrow = Arrow(start=axes.c2p(0, 0), end=end, color=color, buff=0, stroke_width=4)
            lbl = MathTex(label_tex, color=color).scale(0.7).next_to(arrow.get_end(), UR, buff=0.1)
            arrows_orig.append(arrow)
            labels_orig.append(lbl)
            self.play(GrowArrow(arrow), Write(lbl), run_time=0.4)

        self.wait(0.5)

        # 旋转所有向量
        rotations = []
        for i, (vec, color, _) in enumerate(vectors):
            rotated_2d = rot_matrix @ vec
            new_end = axes.c2p(rotated_2d[0], rotated_2d[1])
            arrow = arrows_orig[i]
            rotations.append(Rotate(arrow, angle=angle, about_point=axes.c2p(0, 0)))
            # 更新标签位置
            new_lbl = MathTex(r"\mathbf{v}'_" + str(i+1), color=color).scale(0.7)
            new_lbl.next_to(Arrow(start=axes.c2p(0, 0), end=new_end, buff=0).get_end(), UR, buff=0.1)
            # 先移除新标签再移动（简化处理）
            self.remove(labels_orig[i])

        self.play(*rotations, run_time=2.5)
        self.wait(0.5)


def render_vector_rotation(angle):
    clear_output(wait=True)
    scene = VectorRotationScene(angle_deg=angle)
    scene.render()
    video_path = config.media_dir + "/videos/480p15/MatrixRotation.mp4"
    # video_path = config.media_dir + "/videos/480p15/VectorRotationScene.mp4"
    if os.path.exists(video_path):
        display(Video(video_path, embed=True, width=600))
    else:
        print(f"视频未找到: {video_path}")

# 交互控件
angle_slider2 = widgets.FloatSlider(value=45, min=0, max=360, step=1,
                                     description='旋转角度', continuous_update=False)
out2 = widgets.interactive_output(render_vector_rotation, {'angle': angle_slider2})
display(angle_slider2, out2)

FloatSlider(value=45.0, continuous_update=False, description='旋转角度', max=360.0, step=1.0)

Output()

### 2. 矩阵旋转 (Matrix as Rotation Transformation)

矩阵 $R_\theta$ 本身就是一个**旋转变换**。当它作用在整个平面上时，平面上每一个点 $(x,y)$ 都被映射到新位置 $(x', y')$：

$$\begin{bmatrix} x' \\ y' \end{bmatrix} = 
\begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix}$$

下面展示整个网格被旋转矩阵变换的过程，注意网格**保持正交不变**，只是整体旋转：

In [7]:
class MatrixTransformScene(Scene):
    """矩阵作为旋转变换：展示整个网格被旋转矩阵变换"""
    def __init__(self, angle_deg=45, **kwargs):
        super().__init__(**kwargs)
        self.angle_deg = angle_deg

    def construct(self):
        rad = np.radians(self.angle_deg)
        rot_matrix = np.array([
            [np.cos(rad), -np.sin(rad)],
            [np.sin(rad),  np.cos(rad)]
        ])

        # 坐标轴
        axes = Axes(
            x_range=[-4, 4, 1], y_range=[-4, 4, 1],
            x_length=7, y_length=7,
            axis_config={"include_tip": True, "color": WHITE}
        )
        self.add(axes)

        # 创建原始网格点
        grid_dots = VGroup()
        dot_positions = []
        for x in np.arange(-3.5, 3.6, 0.7):
            for y in np.arange(-3.5, 3.6, 0.7):
                pos = axes.c2p(x, y)
                dot = Dot(point=pos, color=BLUE, radius=0.04)
                grid_dots.add(dot)
                dot_positions.append(np.array([x, y]))

        self.add(grid_dots)

        # 公式
        formula = MathTex(
            r"\begin{bmatrix} x' \\ y' \end{bmatrix} = "
            r"\begin{bmatrix} \cos\theta & -\sin\theta \\ "
            r"\sin\theta & \cos\theta \end{bmatrix}"
            r"\begin{bmatrix} x \\ y \end{bmatrix}"
        ).to_corner(UL).scale(0.7)
        angle_label = MathTex(
            r"\theta = {:.1f}^\circ".format(self.angle_deg)
        ).next_to(formula, DOWN, aligned_edge=LEFT)
        self.add(formula, angle_label)

        self.wait(0.5)

        # 变换所有点到旋转后的位置
        animations = []
        for i, dot in enumerate(grid_dots):
            orig = dot_positions[i]
            rotated = rot_matrix @ orig
            new_pos = axes.c2p(rotated[0], rotated[1])
            animations.append(dot.animate.move_to(new_pos))

        self.play(*animations, run_time=3)
        self.wait(0.5)

        # 添加旋转弧线标注
        arc = Arc(
            radius=1.5, angle=self.angle_deg * DEGREES,
            arc_center=axes.c2p(0, 0), color=YELLOW, stroke_width=3
        )
        self.play(Create(arc), run_time=1)
        self.wait(0.5)


def render_matrix_transform(angle):
    clear_output(wait=True)
    scene = MatrixTransformScene(angle_deg=angle)
    scene.render()
    video_path = config.media_dir + "/videos/480p15/MatrixRotation.mp4"
    # video_path = config.media_dir + "/videos/480p15/MatrixTransformScene.mp4"
    if os.path.exists(video_path):
        display(Video(video_path, embed=True, width=600))
    else:
        print(f"视频未找到: {video_path}")

angle_slider3 = widgets.FloatSlider(value=45, min=0, max=360, step=1,
                                     description='旋转角度', continuous_update=False)
out3 = widgets.interactive_output(render_matrix_transform, {'angle': angle_slider3})
display(angle_slider3, out3)

FloatSlider(value=45.0, continuous_update=False, description='旋转角度', max=360.0, step=1.0)

Output()

### 3. 坐标系旋转 (Coordinate System Rotation)

坐标系旋转是**被动**旋转：坐标轴旋转 $\theta$，向量本身保持不动，但向量的**坐标表示**发生变化。

关键关系：坐标系旋转 $\theta$ 等价于向量**反向**旋转 $-\theta$。

若新坐标系 $O'x'y'$ 由原坐标系 $Oxy$ 逆时针旋转 $\theta$ 得到，则同一点 $P$ 在两坐标系中的坐标关系为：

$$\begin{bmatrix} x' \\ y' \end{bmatrix} = 
\begin{bmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix} = R_{-\theta} \begin{bmatrix} x \\ y \end{bmatrix}$$

下面展示坐标系旋转而向量不动：

In [8]:
class CoordRotationScene(Scene):
    """坐标系旋转：坐标轴旋转，向量不动"""
    def __init__(self, angle_deg=45, **kwargs):
        super().__init__(**kwargs)
        self.angle_deg = angle_deg

    def construct(self):
        angle = self.angle_deg * DEGREES

        # 原始坐标系
        old_axes = Axes(
            x_range=[-3, 3, 1], y_range=[-3, 3, 1],
            x_length=6, y_length=6,
            axis_config={"include_tip": True, "color": WHITE}
        )
        old_labels = old_axes.get_axis_labels(
            MathTex("x").scale(0.8), MathTex("y").scale(0.8)
        )

        self.add(old_axes, old_labels)

        # 固定向量（用虚线表示它们不动）
        vecs_data = [
            (np.array([2, 1]), RED, r"\mathbf{v}"),
            (np.array([-1.2, 2]), GREEN, r"\mathbf{u}"),
        ]
        fixed_arrows = VGroup()
        for vec, color, label in vecs_data:
            end = old_axes.c2p(vec[0], vec[1])
            arrow = Arrow(start=old_axes.c2p(0, 0), end=end, color=color, buff=0, stroke_width=5)
            lbl = MathTex(label, color=color).scale(0.7).next_to(arrow.get_end(), UR, buff=0.1)
            fixed_arrows.add(arrow)
            fixed_arrows.add(lbl)
        self.play(*[GrowArrow(a) for a in fixed_arrows if isinstance(a, Arrow)],
                  *[Write(a) for a in fixed_arrows if isinstance(a, MathTex)],
                  run_time=1)

        self.wait(0.5)

        # 新坐标系（旋转后的轴）
        new_x_end = old_axes.c2p(3 * np.cos(angle), 3 * np.sin(angle))
        new_y_end = old_axes.c2p(-3 * np.sin(angle), 3 * np.cos(angle))

        new_x_axis = Arrow(
            start=old_axes.c2p(0, 0), end=new_x_end,
            color=YELLOW, buff=0, stroke_width=3
        )
        new_y_axis = Arrow(
            start=old_axes.c2p(0, 0), end=new_y_end,
            color=YELLOW, buff=0, stroke_width=3
        )
        new_x_label = MathTex("x'", color=YELLOW).scale(0.8).next_to(new_x_axis.get_end(), DR)
        new_y_label = MathTex("y'", color=YELLOW).scale(0.8).next_to(new_y_axis.get_end(), UL)

        self.play(
            GrowArrow(new_x_axis), GrowArrow(new_y_axis),
            Write(new_x_label), Write(new_y_label),
            run_time=2
        )

        # 旋转弧线
        arc = Arc(
            radius=1.0, angle=angle,
            arc_center=old_axes.c2p(0, 0), color=YELLOW, stroke_width=2
        )
        self.play(Create(arc), run_time=1)

        # 公式
        formula = MathTex(
            r"\begin{bmatrix} x' \\ y' \end{bmatrix} = "
            r"\begin{bmatrix} \cos\theta & \sin\theta \\ "
            r"-\sin\theta & \cos\theta \end{bmatrix}"
            r"\begin{bmatrix} x \\ y \end{bmatrix}"
        ).to_corner(UL).scale(0.65)
        formula_bg = SurroundingRectangle(formula, color=BLACK, fill_opacity=0.8)
        self.add(formula_bg, formula)

        self.wait(1)


def render_coord_rotation(angle):
    clear_output(wait=True)
    scene = CoordRotationScene(angle_deg=angle)
    scene.render()
    video_path = config.media_dir + "/videos/480p15/MatrixRotation.mp4"
    # video_path = config.media_dir + "/videos/480p15/CoordRotationScene.mp4"
    if os.path.exists(video_path):
        display(Video(video_path, embed=True, width=600))
    else:
        print(f"视频未找到: {video_path}")

angle_slider4 = widgets.FloatSlider(value=45, min=0, max=360, step=1,
                                     description='旋转角度', continuous_update=False)
out4 = widgets.interactive_output(render_coord_rotation, {'angle': angle_slider4})
display(angle_slider4, out4)

FloatSlider(value=45.0, continuous_update=False, description='旋转角度', max=360.0, step=1.0)

Output()

### 4. 坐标系变换 (Coordinate Transformation)

坐标系变换是更一般的概念：同一个向量在不同坐标系下有不同的坐标表示。

设基变换矩阵 $P$ 将旧坐标映射到新坐标：$[v]_{new} = P^{-1} [v]_{old}$

对于旋转变换，若新基是旧基旋转 $\theta$ 得到的，则：

$$P = R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$$

$$[v]_{new} = R_\theta^{-1} [v]_{old} = R_{-\theta} [v]_{old}$$

下面同时展示向量在两个坐标系中的坐标：

In [9]:
class CoordTransformScene(Scene):
    """坐标系变换：展示向量在两个坐标系中的不同坐标表示"""
    def __init__(self, angle_deg=45, **kwargs):
        super().__init__(**kwargs)
        self.angle_deg = angle_deg

    def construct(self):
        angle = self.angle_deg * DEGREES
        rad = np.radians(self.angle_deg)

        # 公共原点
        origin = np.array([0, 0, 0])

        # 旧坐标系 (x, y) — 白色
        old_axes_obj = Axes(
            x_range=[-3, 3, 1], y_range=[-3, 3, 1],
            x_length=6, y_length=6,
            axis_config={"include_tip": True, "color": WHITE}
        )
        old_labels = old_axes_obj.get_axis_labels(
            MathTex("x").scale(0.8), MathTex("y").scale(0.8)
        )
        self.add(old_axes_obj, old_labels)

        # 新坐标系 (x', y') — 黄色，旋转后的
        rot = np.array([
            [np.cos(rad), -np.sin(rad), 0],
            [np.sin(rad),  np.cos(rad), 0],
            [0, 0, 0]
        ])
        new_x_end = np.array([3 * np.cos(angle), 3 * np.sin(angle), 0])
        new_y_end = np.array([-3 * np.sin(angle), 3 * np.cos(angle), 0])

        new_x_axis = Arrow(start=origin, end=new_x_end, color=YELLOW, buff=0, stroke_width=3)
        new_y_axis = Arrow(start=origin, end=new_y_end, color=YELLOW, buff=0, stroke_width=3)
        new_x_label = MathTex("x'", color=YELLOW).scale(0.8)
        new_x_label.next_to(new_x_axis.get_end(), DR, buff=0.15)
        new_y_label = MathTex("y'", color=YELLOW).scale(0.8)
        new_y_label.next_to(new_y_axis.get_end(), UL, buff=0.15)

        self.play(
            GrowArrow(new_x_axis), GrowArrow(new_y_axis),
            Write(new_x_label), Write(new_y_label),
            run_time=1.5
        )

        # 选一个向量 v，计算它在两个坐标系中的坐标
        vec_old = np.array([2.5, 1.0])  # 旧坐标系中的坐标
        vec_new = np.array([
            np.cos(rad) * vec_old[0] + np.sin(rad) * vec_old[1],
            -np.sin(rad) * vec_old[0] + np.cos(rad) * vec_old[1]
        ])

        # 在旧坐标系中画向量
        vec_end = old_axes_obj.c2p(vec_old[0], vec_old[1])
        vec_arrow = Arrow(
            start=old_axes_obj.c2p(0, 0), end=vec_end,
            color=RED, buff=0, stroke_width=5
        )
        self.play(GrowArrow(vec_arrow), run_time=1)

        # 在旧坐标系中标注坐标
        coord_old_text = MathTex(
            r"[v]_{{xy}} = ({:.2f},\;{:.2f})".format(vec_old[0], vec_old[1]),
            color=WHITE
        ).to_corner(UL).scale(0.65)
        self.play(Write(coord_old_text), run_time=0.8)

        # 在新坐标系中的坐标 — 投影线
        # x' 分量投影（平行于 y' 轴）
        proj_x_prime_end = old_axes_obj.c2p(
            vec_new[0] * np.cos(rad), vec_new[0] * np.sin(rad)
        )
        proj_y_prime_end = old_axes_obj.c2p(
            -vec_new[1] * np.sin(rad), vec_new[1] * np.cos(rad)
        )

        # 投影虚线
        dashed_to_x = DashedLine(
            start=vec_end, end=proj_x_prime_end, color=YELLOW, stroke_width=1.5
        )
        dashed_to_y = DashedLine(
            start=vec_end, end=proj_y_prime_end, color=YELLOW, stroke_width=1.5
        )
        self.play(Create(dashed_to_x), Create(dashed_to_y), run_time=1.5)

        # 标注新坐标
        coord_new_text = MathTex(
            r"[v]_{{x'y'}} = ({:.2f},\;{:.2f})".format(vec_new[0], vec_new[1]),
            color=YELLOW
        ).next_to(coord_old_text, DOWN, aligned_edge=LEFT).scale(0.65)
        self.play(Write(coord_new_text), run_time=0.8)

        # 变换矩阵
        transform_text = MathTex(
            r"[v]_{{x'y'}} = R_{{-\theta}}\,[v]_{{xy}}",
            color=YELLOW
        ).to_corner(DL).scale(0.65)
        self.play(Write(transform_text), run_time=0.8)

        self.wait(2)


def render_coord_transform(angle):
    clear_output(wait=True)
    scene = CoordTransformScene(angle_deg=angle)
    scene.render()
    video_path = config.media_dir + "/videos/480p15/MatrixRotation.mp4"
    # video_path = config.media_dir + "/videos/480p15/CoordTransformScene.mp4"
    if os.path.exists(video_path):
        display(Video(video_path, embed=True, width=600))
    else:
        print(f"视频未找到: {video_path}")

angle_slider5 = widgets.FloatSlider(value=45, min=10, max=170, step=1,
                                     description='旋转角度', continuous_update=False)
out5 = widgets.interactive_output(render_coord_transform, {'angle': angle_slider5})
display(angle_slider5, out5)

FloatSlider(value=45.0, continuous_update=False, description='旋转角度', max=170.0, min=10.0, step=1.0)

Output()

### 总结对比

| 概念 | 谁在动 | 数学表达 | 变换矩阵 |
|------|--------|----------|----------|
| **向量旋转** (主动) | 向量动，坐标轴不动 | $v' = R_\theta v$ | $R_\theta = \begin{bmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{bmatrix}$ |
| **矩阵旋转** (主动变换) | 整个平面（网格）被映射 | $p' = R_\theta p$ | 同上 $R_\theta$ |
| **坐标系旋转** (被动) | 坐标轴动，向量不动 | $[v]_{new} = R_{-\theta} [v]_{old}$ | $R_{-\theta} = \begin{bmatrix} \cos\theta & \sin\theta \\ -\sin\theta & \cos\theta \end{bmatrix}$ |
| **坐标系变换** (基变换) | 改变基向量，向量不变但坐标变 | $[v]_{B'} = P^{-1} [v]_B$ | $P$ 为基变换矩阵 |

> **核心理解**：向量逆时针旋转 $\theta$ 与 坐标系顺时针旋转 $\theta$，效果相同（坐标表示一致）。这就是主动变换与被动变换的等价性。